### *-----Prerequisites*

* Install ArcGIS Pro Desktop in the local disk.<br/>

    - After installing ArcGIS Pro, check "arcgispro-py3" in the Path.<br/>
        ```Path: C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3.```<br/>

* Create new virtual environment in Anaconda with Arcpy package.<br/>

    - Create an environment with Arcpy package first, do not create environment with python==version<br/>
    - You need to know the path of "arcgispro-py3" in your local disk, mostly you can find it here C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3<br/>
    - Example<br/>
        `conda create --clone "C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3" -n GIS`<br/>
        `conda activate GIS`<br/>
        `conda install -c esri arcpy`<br/>

    - Checking ArcPy installation<br/>
    `Run Python`<br/>
    `Type import arcpy`<br/>
    `If no error message appears, Arcpy is working correctly`<br/>

# *Culvert Implemantation*

In [2]:
# ============================================================
# Session 1. Library Import
# 역할: ArcPy, 경로 처리, 수학 함수 불러오기
# ============================================================
import arcpy                          # ArcGIS geoprocessing 라이브러리
import os                             # 파일/폴더 경로 처리
import math                           # 수학 계산용


# ============================================================
# Session 2. Environment Setup
# 역할: 실행 환경 설정
# ============================================================
arcpy.env.overwriteOutput = True      # 기존 출력 덮어쓰기 허용


# ============================================================
# Session 3. Input / Output Path Setup
# 역할: 입력/출력 경로 정의
# ============================================================
root_folder_in = r"D:\projstore\Junho\Preprocessing\5_HydroUpdating\Culvert_Reburning\Input"
root_folder_out = r"D:\projstore\Junho\Preprocessing\5_HydroUpdating\Culvert_Reburning\Output"

dem = os.path.join(root_folder_in, "DEM", "03160112_masked.tif")                       # 입력 DEM
flowdirection = os.path.join(root_folder_in, "FlowDirection", "03160112_Fdr.tif")     # 입력 Flow Direction
culvert = os.path.join(root_folder_in, "Masked_Crossing", "Masked_Crossing.shp")      # 입력 culvert point
roads = os.path.join(root_folder_in, "Road", "Masked_Roads.shp")                      # 입력 road line

output_polylines = os.path.join(root_folder_out, "culvert_polylines_hybrid.shp")      # 출력 polyline


# ============================================================
# Session 4. User Parameters
# 역할: 사용자 설정값 정의
# ============================================================
target_sr = arcpy.SpatialReference(26916)      # NAD 1983 UTM Zone 16N
segment_length_m = 3.0                         # 선 길이 (한쪽 방향 길이)
diag_offset = segment_length_m / math.sqrt(2)  # 대각선 이동 길이
culvert_id_field = "Crossing_I"               # culvert ID 필드
road_match_tolerance = 2.0                    # culvert-road 매칭 허용 거리 (m)
tangent_sample = 1.0                          # road local tangent 계산용 샘플 길이 (m)


# ============================================================
# Session 5. Utility Function - Ensure Projection
# 역할: 입력 자료를 목표 좌표계로 맞추고 실제 사용할 경로 반환
# ============================================================
def ensure_projected(input_path, out_folder, out_sr):
    desc = arcpy.Describe(input_path)                              # 입력 정보 조회
    sr = desc.spatialReference                                     # 좌표계 객체
    base_name = os.path.splitext(os.path.basename(input_path))[0]  # 파일명 추출

    if desc.dataType in ["ShapeFile", "FeatureClass"]:
        output_path = os.path.join(out_folder, f"{base_name}_utm16.shp")
    elif desc.dataType in ["RasterDataset"]:
        output_path = os.path.join(out_folder, f"{base_name}_utm16.tif")
    else:
        raise ValueError(f"지원하지 않는 데이터 타입입니다: {desc.dataType}")

    if sr is None or sr.name == "Unknown":
        arcpy.AddWarning(f"Unknown projection: {input_path}")
        arcpy.AddWarning("원본 좌표계가 실제 UTM 16N일 때만 DefineProjection이 안전합니다.")
        arcpy.management.DefineProjection(input_path, out_sr)
        return input_path

    if sr.factoryCode == out_sr.factoryCode:
        return input_path

    if desc.dataType in ["ShapeFile", "FeatureClass"]:
        arcpy.management.Project(input_path, output_path, out_sr)
    elif desc.dataType in ["RasterDataset"]:
        arcpy.management.ProjectRaster(input_path, output_path, out_sr)

    return output_path


# ============================================================
# Session 6. Utility Function - FDR Direction Mapping
# 역할: ArcGIS D8 FDR 값을 dx, dy로 변환
# ============================================================
def get_fdr_offsets(fdr_value, straight_len, diag_len):
    direction_map = {
        1:   ( straight_len,  0.0),       # East
        2:   ( diag_len,     -diag_len),  # Southeast
        4:   ( 0.0,          -straight_len),  # South
        8:   (-diag_len,     -diag_len),  # Southwest
        16:  (-straight_len,  0.0),       # West
        32:  (-diag_len,      diag_len),  # Northwest
        64:  ( 0.0,           straight_len),  # North
        128: ( diag_len,      diag_len)   # Northeast
    }
    return direction_map.get(fdr_value, None)


# ============================================================
# Session 7. Utility Function - Normalize Vector
# 역할: 벡터를 단위벡터로 정규화
# ============================================================
def normalize_vector(dx, dy):
    length = math.sqrt(dx * dx + dy * dy)  # 벡터 길이 계산
    if length == 0:
        return None
    return dx / length, dy / length


# ============================================================
# Session 8. Utility Function - Build FDR-Based Line
# 역할: 기존 방식(FDR_ONLY)으로 양방향 선 생성
# ============================================================
def build_fdr_lines(x, y, fdr_value, sr):
    offsets = get_fdr_offsets(fdr_value, segment_length_m, diag_offset)  # FDR 방향 벡터 가져오기
    if offsets is None:
        return None

    dx, dy = offsets                                  # 이동량 분리
    origin = arcpy.Point(x, y)                        # 원점 생성
    flow_end = arcpy.Point(x + dx, y + dy)           # 흐름 방향 종점
    opposite_end = arcpy.Point(x - dx, y - dy)       # 반대 방향 종점

    flow_line = arcpy.Polyline(arcpy.Array([origin, flow_end]), sr)          # FLOW 선
    opposite_line = arcpy.Polyline(arcpy.Array([origin, opposite_end]), sr)  # OPPOSITE 선

    return flow_line, opposite_line


# ============================================================
# Session 9. Utility Function - Local Road Tangent
# 역할: culvert 인근 road local tangent 계산
# ============================================================
def get_local_road_tangent(road_geom, point_geom, sample_dist):
    nearest_measure = road_geom.measureOnLine(point_geom, use_percentage=False)   # point의 도로상 measure 계산
    road_length = road_geom.length                                                # 전체 도로 길이 확인

    if nearest_measure is None:
        return None

    m1 = max(0.0, nearest_measure - sample_dist)                                  # 앞쪽 샘플 위치
    m2 = min(road_length, nearest_measure + sample_dist)                          # 뒤쪽 샘플 위치

    if abs(m2 - m1) < 1e-6:
        return None

    p1 = road_geom.positionAlongLine(m1, use_percentage=False).firstPoint         # 앞 샘플 좌표
    p2 = road_geom.positionAlongLine(m2, use_percentage=False).firstPoint         # 뒤 샘플 좌표

    dx = p2.X - p1.X                                                              # tangent X
    dy = p2.Y - p1.Y                                                              # tangent Y

    return normalize_vector(dx, dy)                                               # 단위벡터 반환


# ============================================================
# Session 10. Utility Function - Hybrid Line Builder
# 역할: road 직교 축 + FDR 방향성으로 hybrid 선 생성
# ============================================================
def build_hybrid_lines(x, y, road_tangent, fdr_value, sr):
    tx, ty = road_tangent                               # road tangent 단위벡터
    culv_dx = -ty                                      # 직교 벡터 X
    culv_dy = tx                                       # 직교 벡터 Y

    fdr_offsets = get_fdr_offsets(fdr_value, 1.0, 1.0 / math.sqrt(2.0))  # 방향 비교용 FDR 단위 벡터
    if fdr_offsets is None:
        return None

    fdr_unit = normalize_vector(fdr_offsets[0], fdr_offsets[1])           # FDR 단위화
    if fdr_unit is None:
        return None

    dot_value = culv_dx * fdr_unit[0] + culv_dy * fdr_unit[1]             # 직교축과 FDR의 내적

    if dot_value < 0:
        culv_dx *= -1.0                                                    # 부호 뒤집기
        culv_dy *= -1.0

    origin = arcpy.Point(x, y)                                             # 원점 생성

    flow_end = arcpy.Point(                                                 # FLOW 종점 생성
        x + culv_dx * segment_length_m,
        y + culv_dy * segment_length_m
    )

    opposite_end = arcpy.Point(                                             # OPPOSITE 종점 생성
        x - culv_dx * segment_length_m,
        y - culv_dy * segment_length_m
    )

    flow_line = arcpy.Polyline(arcpy.Array([origin, flow_end]), sr)         # FLOW 선 생성
    opposite_line = arcpy.Polyline(arcpy.Array([origin, opposite_end]), sr) # OPPOSITE 선 생성

    return flow_line, opposite_line


# ============================================================
# Session 11. Data Validation
# 역할: 입력 파일과 필드 존재 여부 확인
# ============================================================
required_inputs = [dem, flowdirection, culvert, roads]                      # 필수 입력 목록

for input_path in required_inputs:
    if not arcpy.Exists(input_path):
        raise FileNotFoundError(f"입력 파일이 존재하지 않습니다: {input_path}")

culvert_fields = [f.name for f in arcpy.ListFields(culvert)]                # culvert 필드 수집
if culvert_id_field not in culvert_fields:
    raise ValueError(f"필수 필드가 없습니다: {culvert_id_field}")


# ============================================================
# Session 12. Projection Harmonization
# 역할: 모든 입력을 동일 좌표계로 정렬
# ============================================================
os.makedirs(root_folder_out, exist_ok=True)                                 # 출력 폴더 생성

dem_proj = ensure_projected(dem, root_folder_out, target_sr)                # DEM 정렬
flowdirection_proj = ensure_projected(flowdirection, root_folder_out, target_sr)  # FDR 정렬
culvert_proj = ensure_projected(culvert, root_folder_out, target_sr)        # culvert 정렬
roads_proj = ensure_projected(roads, root_folder_out, target_sr)            # road 정렬


# ============================================================
# Session 13. Road Geometry Cache
# 역할: road feature를 메모리에 올려 nearest 탐색에 사용
# ============================================================
road_geometries = []                                                        # road geometry 저장 리스트

with arcpy.da.SearchCursor(roads_proj, ["OID@", "SHAPE@"]) as road_cursor:
    for road_row in road_cursor:
        road_oid = road_row[0]                                              # road OID
        road_geom = road_row[1]                                             # road geometry
        road_geometries.append((road_oid, road_geom))                       # 리스트에 저장


# ============================================================
# Session 14. Output Feature Class Creation
# 역할: 출력 polyline 생성 및 필드 추가
# ============================================================
if arcpy.Exists(output_polylines):
    arcpy.management.Delete(output_polylines)                               # 기존 파일 삭제

arcpy.management.CreateFeatureclass(
    out_path=root_folder_out,
    out_name=os.path.basename(output_polylines),
    geometry_type="POLYLINE",
    spatial_reference=target_sr
)

arcpy.management.AddField(output_polylines, "CROSS_ID", "TEXT", field_length=100)   # culvert ID
arcpy.management.AddField(output_polylines, "FDR", "SHORT")                          # FDR 값
arcpy.management.AddField(output_polylines, "DIR_TYPE", "TEXT", field_length=12)     # FLOW/OPPOSITE
arcpy.management.AddField(output_polylines, "METHOD", "TEXT", field_length=20)       # HYBRID/FDR_ONLY
arcpy.management.AddField(output_polylines, "ROAD_USE", "TEXT", field_length=10)     # YES/NO
arcpy.management.AddField(output_polylines, "ROAD_DIST", "DOUBLE")                    # road까지 거리
arcpy.management.AddField(output_polylines, "ROAD_OID", "LONG")                       # 선택된 road OID
arcpy.management.AddField(output_polylines, "SRC_X", "DOUBLE")                        # 원점 X
arcpy.management.AddField(output_polylines, "SRC_Y", "DOUBLE")                        # 원점 Y


# ============================================================
# Session 15. Main Processing
# 역할: road 연결 여부에 따라 HYBRID 또는 FDR_ONLY 적용
# ============================================================
insert_fields = [
    "SHAPE@", "CROSS_ID", "FDR", "DIR_TYPE",
    "METHOD", "ROAD_USE", "ROAD_DIST", "ROAD_OID", "SRC_X", "SRC_Y"
]

search_fields = ["SHAPE@", culvert_id_field]

created_count = 0
hybrid_count = 0
fallback_count = 0
skipped_nodata = 0
skipped_invalid_fdr = 0

with arcpy.da.InsertCursor(output_polylines, insert_fields) as icur:
    with arcpy.da.SearchCursor(culvert_proj, search_fields) as scur:
        for row in scur:
            point_geom = row[0]                                             # culvert point geometry
            cross_id = str(row[1]) if row[1] is not None else ""            # culvert ID 문자열화

            x = point_geom.firstPoint.X                                     # 원점 X
            y = point_geom.firstPoint.Y                                     # 원점 Y

            cell_value = arcpy.management.GetCellValue(                     # point 위치의 FDR 값 조회
                flowdirection_proj,
                f"{x} {y}"
            ).getOutput(0)

            if cell_value in [None, "", "NoData"]:
                skipped_nodata += 1
                arcpy.AddWarning(f"NoData FDR at culvert {cross_id}")
                continue

            try:
                fdr_value = int(cell_value)                                 # FDR 정수 변환
            except Exception:
                skipped_invalid_fdr += 1
                arcpy.AddWarning(f"Invalid FDR at culvert {cross_id}: {cell_value}")
                continue

            nearest_road_oid = None                                         # nearest road OID 초기화
            nearest_road_geom = None                                        # nearest road geometry 초기화
            nearest_road_dist = None                                        # nearest distance 초기화

            for road_oid, road_geom in road_geometries:                     # 모든 road에 대해 거리 탐색
                dist = point_geom.distanceTo(road_geom)                     # point-road 거리 계산

                if nearest_road_dist is None or dist < nearest_road_dist:
                    nearest_road_dist = dist                                # 최소거리 갱신
                    nearest_road_oid = road_oid                             # nearest road 갱신
                    nearest_road_geom = road_geom                           # nearest geometry 갱신

            method = "FDR_ONLY"                                             # 기본 방법
            road_use = "NO"                                                 # 기본 road 사용 여부

            lines = None                                                    # 생성 선 초기화

            if nearest_road_geom is not None and nearest_road_dist is not None:
                if nearest_road_dist <= road_match_tolerance:               # road와 충분히 가까운 경우
                    tangent = get_local_road_tangent(
                        nearest_road_geom,
                        point_geom,
                        tangent_sample
                    )

                    if tangent is not None:                                 # local tangent 계산 성공 시
                        lines = build_hybrid_lines(
                            x, y, tangent, fdr_value, target_sr
                        )

                        if lines is not None:
                            method = "HYBRID"                               # hybrid 적용
                            road_use = "YES"

            if lines is None:                                               # hybrid 실패 또는 road 미연결 시
                lines = build_fdr_lines(x, y, fdr_value, target_sr)         # 기존 방식 적용
                method = "FDR_ONLY"
                road_use = "NO"

            if lines is None:                                               # 최종적으로도 실패 시 건너뜀
                skipped_invalid_fdr += 1
                continue

            flow_line, opposite_line = lines                                # 두 개의 선 분리

            icur.insertRow([                                                # FLOW 선 저장
                flow_line, cross_id, fdr_value, "FLOW",
                method, road_use, nearest_road_dist, nearest_road_oid, x, y
            ])
            created_count += 1

            icur.insertRow([                                                # OPPOSITE 선 저장
                opposite_line, cross_id, fdr_value, "OPPOSITE",
                method, road_use, nearest_road_dist, nearest_road_oid, x, y
            ])
            created_count += 1

            if method == "HYBRID":
                hybrid_count += 1                                           # hybrid culvert 카운트
            else:
                fallback_count += 1                                         # fallback culvert 카운트


# ============================================================
# Session 16. Final Summary Log
# 역할: 최종 처리 결과 출력
# ============================================================
arcpy.AddMessage("==========================================")
arcpy.AddMessage("Hybrid culvert polyline generation completed.")
arcpy.AddMessage(f"Output: {output_polylines}")
arcpy.AddMessage(f"Created polylines: {created_count}")
arcpy.AddMessage(f"Hybrid culverts: {hybrid_count}")
arcpy.AddMessage(f"Fallback culverts: {fallback_count}")
arcpy.AddMessage(f"Skipped (NoData): {skipped_nodata}")
arcpy.AddMessage(f"Skipped (Invalid FDR): {skipped_invalid_fdr}")
arcpy.AddMessage("==========================================")

print("==========================================")
print("Hybrid culvert polyline generation completed.")
print(f"Output: {output_polylines}")
print(f"Created polylines: {created_count}")
print(f"Hybrid culverts: {hybrid_count}")
print(f"Fallback culverts: {fallback_count}")
print(f"Skipped (NoData): {skipped_nodata}")
print(f"Skipped (Invalid FDR): {skipped_invalid_fdr}")
print("==========================================")

Hybrid culvert polyline generation completed.
Output: D:\projstore\Junho\Preprocessing\5_HydroUpdating\Culvert_Reburning\Output\culvert_polylines_hybrid.shp
Created polylines: 4214
Hybrid culverts: 1725
Fallback culverts: 382
Skipped (NoData): 0
Skipped (Invalid FDR): 0
